# Session 2 · Emotional Health & Well-Being

**R for All of Us Bootcamp · Day 1, 1–3 p.m. ET**

You will score the **PHQ-9** (depression) and **GAD-7** (anxiety) from the Emotional Health History and Well-Being survey, and check that every question ID is the question you think it is.

| Part | Time | What you do |
| --- | --- | --- |
| Demonstration | 1–2 | Pull the survey, check question IDs, score PHQ-9 |
| Lab, Level 1 | 2–2:45 | Check GAD-7 IDs, score GAD-7, join both scales |
| Lab, Level 2 (boss) | if time | Count skipped PHQ-9 items |


### How the game works

- Run each **checkpoint** cell after a task. A pass earns points; a miss prints a tip.
- Stuck? Run `hint("ID")` for a clue. Each hint costs 1 point.
- **Level 1** tasks are for everyone. **Level 2 (boss level)** tasks are optional extras for when you finish early.
- Run `scoreboard()` at any time. At the end of the lab, report your **score only** in the Zoom poll.
- Never paste data, counts or screenshots of participant-level data into the Zoom chat.

## R basics refresher

Session 1 has the full introduction. The short version:

- **Run a cell** with **Shift + Enter**. Run cells from top to bottom. `[*]` means the cell is still running.
- `#` starts a **comment**: R ignores the rest of the line. `# TODO` marks where you write code.
- `install.packages("name")` once (buying the textbook); `library(name)` every time you open the notebook (taking it off the shelf).
- **Red text is not always an error.** `Error` stops the cell; `Warning` and messages (such as the tidyverse "Conflicts" list) do not.
- `x <- value` stores a value in an object named `x`. Names are case-sensitive: `AGECAT` is not `agecat`.
- `c("a", "b")` combines values. Text needs quotes; numbers, `TRUE`, `FALSE`, `NA` and object names do not.
- `df$column` picks a column; `df[rows, columns]` picks rows and columns; `df[df$Age_Survey < 25, ]` keeps the rows where the condition is `TRUE`.
- `==` compares (one `=` is only for arguments); `!=` not equal; `&` and; `|` or; `!` not.
- `x %in% c("a", "b")` is `TRUE` when `x` is any of the listed values. View the documentation for `%in%` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/match.html)
- `NA` means missing. Test for it with `is.na(x)`, never `x == NA`. View the documentation for `is.na()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/NA.html)
- `table(x, useNA = "ifany")` counts each value, including `NA`. View the documentation for `table()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/table.html)
- `nrow(df)` counts rows. View the documentation for `nrow()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/nrow.html)

## Setup

Run these cells once each time you open the notebook. If the second cell stops with an error, run `00_SETUP_C2025Q4R6.ipynb` (Python kernel) first, then restart this kernel.

In [ ]:
library(tidyverse)
library(bigrquery)
options(scipen = 999)   # print long concept codes in full, never as 1.2e+15

In [ ]:
# Load the workspace variables written by 00_SETUP_C2025Q4R6.ipynb
env_file <- "/home/jupyter/Setting_Env_Variables_p2.R"
if (!file.exists(env_file)) {
    stop("Run 00_SETUP_C2025Q4R6.ipynb (Python kernel) first, then restart this kernel.")
}
source(env_file)

BILLING <- Sys.getenv("GOOGLE_CLOUD_PROJECT")   # your project: it pays for each query
CDR     <- Sys.getenv("WORKSPACE_CDR")          # the All of Us data, e.g. <project>.C2025Q4R6
BUCKET  <- Sys.getenv("WORKSPACE_BUCKET")       # your workspace bucket (gs://...)

if (any(c(BILLING, CDR, BUCKET) %in% c("", "NOT_FOUND"))) {
    stop("A workspace variable is missing. Re-run 00_SETUP_C2025Q4R6.ipynb, then restart this kernel.")
}

In [ ]:
# ---- Game helpers: run once, no need to read ----
score <- 0
earned <- c()
hints_used <- c()

check <- function(id, ok, points, success, tip) {
    ok <- tryCatch(isTRUE(ok), error = function(e) FALSE)
    if (ok) {
        if (!(id %in% earned)) { score <<- score + points; earned <<- c(earned, id) }
        cat(sprintf("PASS     [%s] %s  (+%d)   Score: %d\n", id, success, points, score))
    } else {
        cat(sprintf("NOT YET  [%s] %s\n", id, tip))
    }
    invisible(ok)
}

hints <- list(
    "2.A" = "The query should return 9 rows, and every concept_name should start with 'Over the last 2 weeks'.",
    "2.B" = "rowwise() + sum(c_across(PHQ1:PHQ9)) adds the 9 items. Each item is 0-3, so totals run 0-27.",
    "2.C" = "Copy the PHQ check cell and swap phq for gadq. Expect 7 rows.",
    "2.D" = "Copy the GAD1 line six times and change both numbers: GAD2 = score_item(GAD2A), and so on up to GAD7.",
    "2.E" = "GAD-7 bands: 0-4 Minimal, 5-9 Mild, 10-14 Moderate, 15-21 Severe.",
    "2.F" = "full_join() keeps people who answered only one of the two scales.",
    "2.G" = "rowSums(is.na(across(PHQ1:PHQ9))) counts the skipped items per row."
)

hint <- function(id) {
    if (!(id %in% hints_used)) { score <<- score - 1; hints_used <<- c(hints_used, id) }
    cat(sprintf("HINT [%s] %s\n(-1 point)   Score: %d\n", id, hints[[id]], score))
}

scoreboard <- function() {
    cat(sprintf("Score: %d   Checkpoints passed: %s\n", score,
                if (length(earned)) paste(earned, collapse = ", ") else "none yet"))
}

# Checks used by the checkpoints
one_row_per_person <- function(df) nrow(df) > 0 && !any(duplicated(df$person_id))
only_values <- function(x, allowed) length(x) > 0 && all(na.omit(as.character(x)) %in% allowed)
in_range <- function(x, low, high) any(!is.na(x)) && all(na.omit(x) >= low & na.omit(x) <= high)

In [ ]:
# ---- Save to and load from your workspace bucket ----
save_to_bucket <- function(df, filename) {
    dir.create("data", showWarnings = FALSE)
    local_file <- file.path("data", filename)
    write_excel_csv(df, local_file)
    system(paste0("gsutil cp ", local_file, " ", BUCKET, "/data/"), intern = TRUE)
    message("Saved ", filename, " to ", BUCKET, "/data/")
}

load_from_bucket <- function(filename) {
    dir.create("data", showWarnings = FALSE)
    system(paste0("gsutil cp ", BUCKET, "/data/", filename, " data/"), intern = TRUE)
    read_csv(file.path("data", filename), show_col_types = FALSE)
}

## Study cohort: participants born in 1975 or later

Every query in this notebook keeps only participants **born in 1975 or later**. Here is why we define the cohort by year of birth.

**1. The data have a cutoff date.** CDR v9 (`C2025Q4R6`) "includes participant data with a cutoff date of January 1, 2025"
([CDR v9 release notes](https://support.researchallofus.org/hc/en-us/articles/50653909888788-Curated-Data-Repository-CDR-version-9-Release-Notes)). Data curated after that date are not in this release.
The cutoff is when curation stopped, not a promise that every person has records up to that day.
Cutoff dates for every release are in the [Data Dictionaries](https://support.researchallofus.org/hc/en-us/articles/360033200232-Data-Dictionaries-for-the-Curated-Data-Repositories-CDRs).

**2. Dates of birth are protected.** All of Us changes dates to protect participants' privacy
([Participant Privacy Protections](https://support.researchallofus.org/hc/en-us/articles/4552681983764-Participant-Privacy-Protections); [Data Curation Questions](https://support.researchallofus.org/hc/en-us/articles/35017475171348)):

| Tier | Dates of events (surveys, measurements, diagnoses) | Date of birth |
| --- | --- | --- |
| **Registered** (`R...` datasets) | Shifted backwards by a random 1 to 365 days, the same shift for every date of a participant | Shifted by the same amount |
| **Controlled** (`C...` datasets, used here) | Real dates | **Generalized to year of birth** |

In the Controlled Tier, the year of birth is real, but the month and day of `birth_datetime` are not the participant's.
(Session 1 shows which month and day are stored.)

**3. So we use the year, and only the year.** Born in 1975 or later means a participant was **at most 49 at any point in 2024**,
the last full year before the cutoff. Using one fixed birth year keeps the same people in all four notebooks, whenever the code is run.
A cutoff written as a full date (for example, born after 1975-01-01) would depend on a month and day that are not real.

**4. Ages are computed from years too.** Age at an event = the event's year minus the year of birth. This is the age the person
reaches during that calendar year: it equals their true age, or is one year more if their birthday comes later that year.
It is **never lower** than the true age. The next note explains why that matters.

**Why not subtract dates?** If we subtracted the stored `birth_datetime` from an event date, the answer would depend on the stored month and day.
If that stored day falls later in the year than a participant's real birthday, the computed age comes out too young,
by up to a year. All of Us enrolls adults **18 years or older** ([Join All of Us](https://az.joinallofus.org/en/get-started)),
yet a participant who enrolled shortly after turning 18 could then appear to be **17** at their first survey.
In the Registered Tier the birth date and event dates move by the same shift, so date differences keep the right age,
but the dates themselves no longer match the calendar, so a fixed calendar cutoff would still be off by up to a year.
Year arithmetic avoids both problems: an adult who was 18 at enrollment always comes out as 18 or older.

The filter runs inside BigQuery, so only these participants' rows ever leave Google's servers. That makes every query smaller and faster.

**New in the next cell**

- SQL a query language that BigQuery runs on Google's servers; only the result comes back to your notebook. `SELECT` picks columns, `FROM` names a table, `JOIN ... ON` links two tables on a shared column, `WHERE` keeps the rows that match, and `IN (...)` matches any value in a list or subquery.  
  View the documentation for SQL by clicking [**here.**](https://cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax)
- `str_glue()` builds text and fills in anything in `{curly braces}` with the value of that R object. We use it to put `CDR`, `FIRST_BIRTH_YEAR` and `UNDER50` into SQL.  
  View the documentation for `str_glue()` by clicking [**here.**](https://stringr.tidyverse.org/reference/str_glue.html)

In [ ]:
CUTOFF <- "2025-01-01"          # CDR v9 data cutoff (release notes)
FIRST_BIRTH_YEAR <- 1975     # born 1975 or later: at most 49 at any point in 2024

# A subquery: the person_id of everyone born in FIRST_BIRTH_YEAR or later.
# year_of_birth is real in the Controlled Tier; the month and day of birth_datetime are not.
# Other queries use it as  WHERE person_id IN ({UNDER50})
UNDER50 <- str_glue("
    SELECT person_id
    FROM `{CDR}.person`
    WHERE year_of_birth >= {FIRST_BIRTH_YEAR}")

In [ ]:
# ---- Run a query once, keep the result in your bucket (run once, no need to read) ----
# First run: BigQuery runs the SQL and saves the result as CSV files in your bucket.
# Later runs: the saved files are read back, with no new query (faster, and no query cost).
# After you change a query, use refresh = TRUE to run it again.
bq_to_df <- function(sql, name, refresh = FALSE) {
    path  <- file.path(BUCKET, "bq_exports", name, paste0(name, "_*.csv"))
    files <- suppressWarnings(system2("gsutil", c("ls", path), stdout = TRUE, stderr = FALSE))
    if (refresh || length(files) == 0 || !is.null(attr(files, "status"))) {
        message("Running the query for ", name, " in BigQuery...")
        bq_table_save(bq_project_query(BILLING, sql), path, destination_format = "CSV")
        files <- system2("gsutil", c("ls", path), stdout = TRUE)
    } else {
        message("Reading saved results for ", name, " (refresh = TRUE re-runs the query)")
    }
    bind_rows(lapply(files, function(f)
        read_csv(pipe(paste("gsutil cat", f)), col_types = cols(.default = col_character()),
                 show_col_types = FALSE)))
}

# Part A · Demonstration

## A1. The question IDs

| Scale | Item concept IDs |
| --- | --- |
| PHQ-9 | 1704026, 1704024, 1703983, 1704039, 1704004, 1704041, 1704038, 1703996, 1703977 |
| GAD-7 | 1703984, 1703995, 1704000, 1703987, 1704028, 1703920, 1704042 |

In [ ]:
phq  <- c(1704026, 1704024, 1703983, 1704039, 1704004, 1704041, 1704038, 1703996, 1703977)
gadq <- c(1703984, 1703995, 1704000, 1703987, 1704028, 1703920, 1704042)

## A2. Query check: are these really the PHQ-9 questions?

Before using any list of concept IDs (from a colleague, an old notebook or an AI assistant), look them up in the `concept` table. A mistyped or made-up ID returns no row. This query is tiny, so it runs in seconds.

In [ ]:
phq_sql <- str_glue("SELECT concept_id, concept_name FROM `{CDR}.concept`
                     WHERE concept_id IN ({paste(phq, collapse = ', ')})")
phq_concepts <- bq_table_download(bq_project_query(BILLING, phq_sql))

phq_concepts

In [ ]:
check("2.A", nrow(phq_concepts) == 9 && all(grepl("Over the last 2 weeks", phq_concepts$concept_name)),
      5, "All 9 PHQ-9 question IDs exist and match.", "Expected 9 rows that start with 'Over the last 2 weeks'.")

## A3. Get the answers (only the 16 items we need)

**New in the next cell**

- `%>%` (the pipe) passes the result on its left into the function on its right, so steps read top to bottom. Read it as "and then": `df %>% filter(...) %>% mutate(...)`.  
  View the documentation for `%>%` by clicking [**here.**](https://magrittr.tidyverse.org/reference/pipe.html)
- `mutate()` adds a new column or changes an existing one. Every row is kept.  
  View the documentation for `mutate()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/mutate.html)

In [ ]:
# Emotional Health History and Well-Being answers, from Data Explorer's survey index.
# Rewritten from the Data Explorer export: it asks only for the 16 PHQ-9 and GAD-7 items
# and only for the under-50 cohort, instead of every item in the survey.
# If BigQuery says the index dataset is "Not found", export the survey again from Data Explorer
# and copy the dataset name it uses into INDEX_DATASET.
INDEX_DATASET <- paste0(CDR, "_index_061026")

item_list <- paste0("'", c(phq, gadq), "'", collapse = ", ")

survey_sql <- str_glue("
    SELECT person_id,
           CAST(question AS STRING) AS question_id,
           T_DISP_answer AS answer,
           CAST(survey_datetime AS DATE) AS observation_date,
           survey_datetime AS observation_datetime
    FROM `{INDEX_DATASET}.T_ENT_surveyOccurrence`
    WHERE CAST(question AS STRING) IN ({item_list})
      AND person_id IN ({UNDER50})")

df <- bq_to_df(survey_sql, "s2_phq_gad_yob1975") %>%
    mutate(person_id = as.numeric(person_id),
           question_id = as.numeric(question_id))

In [ ]:
head(df)

## A4. Score the PHQ-9

1. Keep the 9 items and reshape **long to wide**: one row per person per survey date.
2. Turn each answer into 0–3.
3. Add them up. If any item was skipped, the total is `NA`: that's a decision, so say it out loud.
4. Keep each person's earliest survey.

**New in the next cell**

- `filter()` keeps only the rows where the condition is `TRUE`.  
  View the documentation for `filter()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/filter.html)
- `distinct()` removes duplicate rows. With column names, it keeps one row per combination of those columns (`.keep_all = TRUE` keeps the other columns too).  
  View the documentation for `distinct()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/distinct.html)
- `arrange()` sorts the rows. `desc()` sorts largest or newest first.  
  View the documentation for `arrange()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/arrange.html)
- `group_by()` splits the rows into groups (here, one group per person) so the next step works within each group.  
  View the documentation for `group_by()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/group_by.html)
- `slice()` keeps rows by position. After `group_by()`, `slice(1)` keeps the first row in each group and `slice(1:3)` keeps up to three.  
  View the documentation for `slice()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/slice.html)
- `ungroup()` removes the grouping, so later steps work on the whole table again. Forgetting it is a common source of surprises.  
  View the documentation for `ungroup()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/group_by.html)
- `as.Date()` turns text such as `"2023-08-01"` into a date that R can sort, compare and subtract. `format = "%Y-%m-%d"` tells R the layout (year-month-day).  
  View the documentation for `as.Date()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/as.Date.html)
- `case_when()` a list of `condition ~ value` rules, checked in order; each row gets the value of the first rule that is `TRUE`. `.default` is the value when no rule matches.  
  View the documentation for `case_when()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/case_when.html)
- `select()` keeps only the columns you name, in the order you name them.  
  View the documentation for `select()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/select.html)
- `pivot_wider()` reshapes long data (one row per person per question) into wide data (one row per person, one column per question). `names_from` gives the new column names, `values_from` fills them.  
  View the documentation for `pivot_wider()` by clicking [**here.**](https://tidyr.tidyverse.org/reference/pivot_wider.html)

In [ ]:
score_item <- function(x) case_when(
    x == "Not at all" ~ 0,
    x == "Several days" ~ 1,
    x == "More than half the days" ~ 2,
    x == "Nearly every day" ~ 3,
    .default = NA)

phqdf <- df %>%
    select(person_id, question_id, answer, observation_date, observation_datetime) %>%
    filter(question_id %in% phq) %>%
    pivot_wider(id_cols = c(person_id, observation_date, observation_datetime),   # one row per survey taken
                names_from = question_id,
                names_prefix = "Q",
                values_from = answer) %>%
    rename("PHQ1A" = "Q1704026",
           "PHQ2A" = "Q1704024",
           "PHQ3A" = "Q1703983",
           "PHQ4A" = "Q1704039",
           "PHQ5A" = "Q1704004",
           "PHQ6A" = "Q1704041",
           "PHQ7A" = "Q1704038",
           "PHQ8A" = "Q1703996",
           "PHQ9A" = "Q1703977") %>%
    mutate(
        PHQ1 = score_item(PHQ1A), PHQ2 = score_item(PHQ2A), PHQ3 = score_item(PHQ3A),
        PHQ4 = score_item(PHQ4A), PHQ5 = score_item(PHQ5A), PHQ6 = score_item(PHQ6A),
        PHQ7 = score_item(PHQ7A), PHQ8 = score_item(PHQ8A), PHQ9 = score_item(PHQ9A)) %>%
    rowwise() %>%
    mutate(PHQSUM = sum(c_across(PHQ1:PHQ9), na.rm = FALSE)) %>%   # NA if any item was skipped
    ungroup() %>%
    distinct() %>%
    mutate(PHQdt = as.Date(observation_date, format = "%Y-%m-%d")) %>%
    arrange(person_id, PHQdt) %>%
    group_by(person_id) %>%
    slice(1) %>%                                                  # earliest survey per person
    ungroup()

nrow(phqdf)
any(duplicated(phqdf$person_id))

In [ ]:
# Severity bands: 0-4 minimal, 5-9 mild, 10-14 moderate, 15-19 moderately severe, 20-27 severe
phqdf <- phqdf %>%
    mutate(PHQCAT = case_when(
        PHQSUM <= 4 ~ "Minimal",
        PHQSUM <= 9 ~ "Mild",
        PHQSUM <= 14 ~ "Moderate",
        PHQSUM <= 19 ~ "Moderately severe",
        PHQSUM <= 27 ~ "Severe"
    ))

table(phqdf$PHQCAT, useNA = "ifany")

In [ ]:
check("2.B", one_row_per_person(phqdf) && in_range(phqdf$PHQSUM, 0, 27),
      5, "PHQ-9 scored: totals 0-27, one row per person.", "Totals outside 0-27 or repeated person_id.")

# Part B · Coding lab

## Level 1

### Task 2.C · Check the GAD-7 question IDs

In [ ]:
# TASK 2.C: are these the 7 GAD-7 questions? Write the check yourself (copy the PHQ version above).
gad_concepts <- NULL   # TODO

gad_concepts

In [ ]:
check("2.C", nrow(gad_concepts) == 7 && all(grepl("Over the last 2 weeks", gad_concepts$concept_name)),
      10, "All 7 GAD-7 question IDs exist and match.", "Expected 7 rows that start with 'Over the last 2 weeks'.")

### Task 2.D · Score the GAD-7

**New in Task 2.D**

- `summary()` a quick look at a numeric column: minimum, quartiles, median, mean, maximum and the number of NAs.  
  View the documentation for `summary()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/summary.html)

In [ ]:
# TASK 2.D: score the GAD-7, one row per person (earliest survey)
gaddf <- df %>%
    select(person_id, question_id, answer, observation_date, observation_datetime) %>%
    filter(question_id %in% gadq) %>%
    pivot_wider(id_cols = c(person_id, observation_date, observation_datetime),   # one row per survey taken
                names_from = question_id,
                names_prefix = "Q",
                values_from = answer) %>%
    rename("GAD1A" = "Q1703984",
           "GAD2A" = "Q1703995",
           "GAD3A" = "Q1704000",
           "GAD4A" = "Q1703987",
           "GAD5A" = "Q1704028",
           "GAD6A" = "Q1703920",
           "GAD7A" = "Q1704042") %>%
    mutate(
        GAD1 = score_item(GAD1A),
        # TODO: GAD2 to GAD7
    ) %>%
    rowwise() %>%
    mutate(GADSUM = sum(c_across(GAD1:GAD7), na.rm = FALSE)) %>%
    ungroup() %>%
    mutate(GADdt = as.Date(observation_date, format = "%Y-%m-%d")) %>%
    arrange(person_id, GADdt) %>%
    group_by(person_id) %>%
    slice(1) %>%
    ungroup()

summary(gaddf$GADSUM)

In [ ]:
check("2.D", one_row_per_person(gaddf) && in_range(gaddf$GADSUM, 0, 21) &&
             all(paste0("GAD", 1:7) %in% names(gaddf)),
      15, "GAD-7 scored: totals 0-21, one row per person.", "Make all 7 items (GAD1 to GAD7) before adding them up.")

### Task 2.E · GAD-7 severity

In [ ]:
# TASK 2.E: GAD-7 severity
gaddf <- gaddf %>%
    mutate(GADCAT = case_when(
        # TODO: 0-4 "Minimal", 5-9 "Mild", 10-14 "Moderate", 15-21 "Severe"
    ))

table(gaddf$GADCAT, useNA = "ifany")

In [ ]:
check("2.E", only_values(gaddf$GADCAT, c("Minimal", "Mild", "Moderate", "Severe")) &&
             any(!is.na(gaddf$GADCAT)) && "Minimal" %in% gaddf$GADCAT,
      10, "GAD-7 severity bands assigned.", "Use the bands 0-4, 5-9, 10-14 and 15-21.")

### Task 2.F · Join PHQ-9 and GAD-7

This code is given. Run it, read it, then run the checkpoint.

In [ ]:
totdf <- gaddf %>%
    select(person_id, GAD1:GAD7, GADSUM, GADCAT, GADdt) %>%
    full_join(phqdf %>% select(person_id, PHQ1:PHQ9, PHQSUM, PHQCAT, PHQdt), by = "person_id") %>%
    mutate(EHHWBdt = coalesce(GADdt, PHQdt))

nrow(totdf)

In [ ]:
check("2.F", one_row_per_person(totdf) && all(c("PHQSUM", "GADSUM", "EHHWBdt") %in% names(totdf)),
      10, "PHQ-9 and GAD-7 joined. Level 1 cleared!", "person_id repeats after the join. Check gaddf and phqdf first.")
save_to_bucket(totdf, "S2_PHQ_GAD.csv")

## Level 2 (boss level)

### Task 2.G · Skipped items

**New in Task 2.G**

- `ifelse(test, yes, no)` gives `yes` where the test is `TRUE` and `no` where it is `FALSE`, row by row.  
  View the documentation for `ifelse()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/ifelse.html)
- `across()` applies the same function to several columns inside `mutate()` or `summarise()`.  
  View the documentation for `across()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/across.html)

In [ ]:
# BOSS TASK 2.G: how many PHQ-9 items did each person skip? Make PHQ_NMISS (0 to 9) in totdf.
totdf <- totdf %>%
    mutate(PHQ_NMISS = NA)   # TODO

table(totdf$PHQ_NMISS, useNA = "ifany")

In [ ]:
check("2.G", in_range(totdf$PHQ_NMISS, 0, 9) && any(totdf$PHQ_NMISS > 0, na.rm = TRUE),
      20, "Boss defeated: skipped items counted.", "PHQ_NMISS should be a count from 0 to 9.")
save_to_bucket(totdf, "S2_PHQ_GAD.csv")

## Finish line

In [ ]:
scoreboard()
cat("\nReport your score (only your score) in the Zoom poll for Session 2.\n")